# 🔱 Shiv AI Voice Cloning v3.0
**Owner: Shri Ram Nag | PAISAWALA Channel**

### ✅ v3 Fix: Long Hindi Script Chunking Bug Fixed!
- **Problem था:** Script aadha padhta tha, bich mein ruk jaata tha, 2 min baad dobara padhta tha
- **Fix:** `split_chunks_v2()` — Newline-based smart merging, ellipsis preserve, 90-char optimal chunks

### Features:
- 🎙️ **Voice Clone** — Reference audio se aawaz clone
- 🎛️ **Voice Design** — Speed, Pitch, Energy, Pause + Presets
- 🔤 **Simple TTS** — Seedha text se audio
- 📋 **Instruct Mode** — Instructions se voice style

> ⚡ **Runtime → Change runtime type → T4 GPU** pehle select karein!

In [ ]:
# ✅ STEP 1: GPU Check
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 2), 'GB')
else:
    raise RuntimeError('❌ GPU nahi mila! Runtime → Change runtime type → T4 GPU!')

In [ ]:
# ✅ STEP 2: Install Dependencies
!pip install -q gradio huggingface_hub transformers accelerate scipy numpy
print('✅ Done!')

In [ ]:
# ✅ STEP 3: Download Model from HuggingFace
import os
from huggingface_hub import snapshot_download

REPO_ID  = 'Shriramnag/Shiv-AI-Voice-Cloning'
LOCAL    = './Shiv-AI-Voice-Cloning'

if not os.path.exists(LOCAL) or not os.listdir(LOCAL):
    print(f'📥 Downloading {REPO_ID} (~3.27 GB)...')
    snapshot_download(repo_id=REPO_ID, local_dir=LOCAL, local_dir_use_symlinks=False)
    print('✅ Download complete!')
else:
    print('✅ Already downloaded!')

for f in sorted(os.listdir(LOCAL)):
    sz = os.path.getsize(os.path.join(LOCAL,f))/1e6
    print(f'  {f:40s} {sz:.2f} MB')

In [ ]:
# ✅ STEP 4: Imports & Model Load
import sys, re, logging
import numpy as np
import torch
import gradio as gr

MODEL_PATH = './Shiv-AI-Voice-Cloning'
sys.path.insert(0, MODEL_PATH)

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.lang_map import LANG_NAMES, lang_display_name
try:
    from subtitle import LANGUAGE_CODE as WHISPER_LANGUAGE_CODE
except ImportError:
    WHISPER_LANGUAGE_CODE = None

print('🔱 Loading Shiv AI...')
model = OmniVoice.from_pretrained(MODEL_PATH, device_map='cuda', dtype=torch.float16, load_asr=False)
SR    = model.sampling_rate
print(f'✅ Model loaded! SR={SR}')

In [ ]:
# ✅ STEP 5: Fixed Chunking + All Functions
os.makedirs('./Shiv_Audio', exist_ok=True)

LANG_CHOICES = ['Auto'] + sorted(lang_display_name(n) for n in LANG_NAMES)
EVENT_TAGS   = ['[laughter]','[sigh]','[confirmation-en]','[question-en]','[surprise-wa]','[dissatisfaction-hnn]']
INSTRUCT_EX  = [
    'Speak slowly and clearly with a calm, deep voice',
    'Speak with excitement and high energy',
    'Speak softly like a bedtime story narrator',
    'Speak like a professional news anchor, formal and clear',
    'Speak in a sad, emotional tone with pauses',
    'Fast and enthusiastic like a radio jockey',
    'धीरे, शांत और गहरी आवाज़ में बोलें',
    'जोश और उत्साह के साथ तेज़ आवाज़ में बोलें',
]
INSERT_TAG_JS = """
(tag_val, cur) => {
    const ta = document.querySelector('.shiv-tb textarea');
    if (!ta) return cur + ' ' + tag_val;
    const s = ta.selectionStart, e = ta.selectionEnd;
    return cur.slice(0,s) + ' ' + tag_val + ' ' + cur.slice(e);
}
"""

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 🔧 FIXED CHUNKING v2 — Long Script Bug Fix
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def split_chunks(text, max_ch=90):
    """
    v2 Fix:
    - Newline pe split karo (…  ellipsis preserve — dramatic pause)
    - Short lines ko merge karo 90-char tak (model ke liye optimal)
    - Badi lines ko sentence boundary pe toddo
    - Koi bhi chunk drop NAHI hoga (purana bug: bich ke chunks gayab ho jaate the)
    """
    raw_lines = text.split('\n')
    lines = [l.strip() for l in raw_lines if l.strip()]
    chunks, cur = [], ''
    for line in lines:
        if len(line) > max_ch:
            if cur:
                chunks.append(cur)
                cur = ''
            parts = re.split(r'(?<=[।.!?,…])\s*', line)
            sub = ''
            for p in parts:
                p = p.strip()
                if not p: continue
                if len(sub)+len(p)+1 <= max_ch:
                    sub = (sub+' '+p).strip() if sub else p
                else:
                    if sub: chunks.append(sub)
                    sub = p
            if sub: chunks.append(sub)
        else:
            merged = (cur+' '+line).strip() if cur else line
            if len(merged) <= max_ch:
                cur = merged
            else:
                if cur: chunks.append(cur)
                cur = line
    if cur: chunks.append(cur)
    return [c for c in chunks if c.strip()]

def join_audio(audios, sil_ms=0):
    if sil_ms > 0:
        sil = np.zeros(int(SR*sil_ms/1000), dtype=np.float32)
        out = []
        for i,a in enumerate(audios):
            out.append(a)
            if i < len(audios)-1: out.append(sil)
        return np.concatenate(out)
    return np.concatenate(audios)

def make_cfg(steps=32, gs=2.0, speed=1.0, pitch=0, energy=1.0):
    try:
        return OmniVoiceGenerationConfig(num_step=steps, guidance_scale=gs,
            denoise=True, preprocess_prompt=True, postprocess_output=True,
            speed=speed, pitch=pitch, energy=energy)
    except TypeError:
        return OmniVoiceGenerationConfig(num_step=steps, guidance_scale=gs,
            denoise=True, preprocess_prompt=True, postprocess_output=True)

def run_chunk(text, lang, cfg, vcp=None, inst=None):
    kw = dict(text=text, language=lang if lang!='Auto' else None, generation_config=cfg)
    if vcp:  kw['voice_clone_prompt'] = vcp
    if inst: kw['instruct'] = inst
    return model.generate(**kw)[0]

def to_wav(a): return (SR, (a*32767).astype(np.int16))

# ── Tab Functions ─────────────────────────────────────────────────────
def fn_clone(text, lang, ref, ref_text):
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    if not ref:                      return None, '⚠️ Reference audio upload karein'
    try:
        vcp    = model.create_voice_clone_prompt(ref_audio=ref, ref_text=ref_text.strip() or None)
        chunks = split_chunks(text)
        print(f'📦 {len(chunks)} chunks bane')
        audio  = join_audio([run_chunk(c,lang,make_cfg(),vcp=vcp) for c in chunks])
        return to_wav(audio), f'✅ Done! {len(chunks)} chunks | {len(audio)/SR:.1f}s'
    except Exception as e: return None, f'❌ {e}'

def fn_design(text, lang, speed, pitch, energy, pause_ms, style):
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    try:
        cfg    = make_cfg(gs=2.5, speed=speed, pitch=pitch, energy=energy)
        inst   = style.strip() or None
        chunks = split_chunks(text)
        print(f'📦 {len(chunks)} chunks bane')
        audio  = join_audio([run_chunk(c,lang,cfg,inst=inst) for c in chunks], sil_ms=int(pause_ms))
        return to_wav(audio), f'✅ Done! speed={speed} pitch={pitch} energy={energy} | {len(audio)/SR:.1f}s'
    except Exception as e:
        try:
            chunks = split_chunks(text)
            audio  = join_audio([run_chunk(c,lang,make_cfg(gs=2.5),inst=style.strip() or None) for c in chunks])
            return to_wav(audio), f'✅ Basic mode | {len(audio)/SR:.1f}s'
        except Exception as e2: return None, f'❌ {e2}'

def fn_tts(text, lang, steps, gs):
    if not text or not text.strip(): return None, '⚠️ Text likhein'
    try:
        chunks = split_chunks(text)
        print(f'📦 {len(chunks)} chunks bane')
        audio  = join_audio([run_chunk(c,lang,make_cfg(int(steps),float(gs))) for c in chunks])
        return to_wav(audio), f'✅ Done! {len(chunks)} chunks | {len(audio)/SR:.1f}s'
    except Exception as e: return None, f'❌ {e}'

def fn_instruct(text, lang, prompt):
    if not text   or not text.strip():   return None, '⚠️ Text likhein'
    if not prompt or not prompt.strip(): return None, '⚠️ Instruction likhein'
    try:
        chunks = split_chunks(text)
        print(f'📦 {len(chunks)} chunks bane')
        audio  = join_audio([run_chunk(c,lang,make_cfg(gs=3.0),inst=prompt.strip()) for c in chunks])
        return to_wav(audio), f'✅ Done! {len(chunks)} chunks | {len(audio)/SR:.1f}s'
    except Exception as e: return None, f'❌ {e}'

print('✅ All functions ready!')

# Quick chunking test
test = 'रुकिए…\nएक पल के लिए रुकिए।\nयह एक ऐसा सच है…\nजो आपकी सोच को हिला सकता है।'
tc = split_chunks(test)
print(f'🧪 Test: {len(tc)} chunks — {tc}')

In [ ]:
# ✅ STEP 6: Launch Gradio UI
theme = gr.themes.Soft(primary_hue='orange', font=['Inter','Arial','sans-serif'])
css = """
.gradio-container{max-width:100%!important;}
footer{display:none!important;}
.shiv-header{text-align:center;padding:24px;border-bottom:2px solid #ff6600;}
.tag-btn{background:#fff3e0!important;border:1px solid #ffcc80!important;color:#e65100!important;font-size:0.8em!important;}
"""

with gr.Blocks(theme=theme, css=css, title='🔱 Shiv AI Voice Cloning') as demo:
    gr.HTML("""
        <div class='shiv-header'>
            <h1 style='font-size:2.5em;color:#ff6600;margin:0;'>🔱 Shiv AI Voice Cloning</h1>
            <p style='color:#555;'><b>Owner: Shri Ram Nag</b> | PAISAWALA 🎬 | v3.0 — Long Script Fix ✅</p>
            <p style='color:#888;font-size:.85em;'>Model: Shriramnag/Shiv-AI-Voice-Cloning | 646 Languages</p>
        </div>
    """)

    with gr.Tabs():

        # ── TAB 1: Voice Clone ─────────────────────────────────────────
        with gr.TabItem('🎙️ Voice Clone'):
            with gr.Row():
                with gr.Column():
                    vc_text = gr.Textbox(label='📝 Text', lines=8, elem_classes='shiv-tb',
                                         placeholder='यहाँ पूरी script paste करें — long script bhi chalegi!')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b.click(fn=None, inputs=[b,vc_text], outputs=vc_text, js=INSERT_TAG_JS)
                    vc_lang     = gr.Dropdown(label='🌐 Language', choices=LANG_CHOICES, value='Auto')
                    vc_ref      = gr.Audio(label='🎤 Reference Audio', type='filepath')
                    vc_ref_text = gr.Textbox(label='📄 Reference Transcript (optional)', lines=2)
                    vc_btn      = gr.Button('🔱 Clone Voice', variant='primary', size='lg')
                with gr.Column():
                    vc_out    = gr.Audio(label='🔊 Output', type='numpy')
                    vc_status = gr.Textbox(label='Status', interactive=False)
                    gr.Markdown('**💡 Long script tip:** Puri script paste karein, system automatically sahi chunks mein todega!')
            vc_btn.click(fn_clone, [vc_text,vc_lang,vc_ref,vc_ref_text], [vc_out,vc_status])

        # ── TAB 2: Voice Design ────────────────────────────────────────
        with gr.TabItem('🎛️ Voice Design'):
            with gr.Row():
                with gr.Column():
                    vd_text   = gr.Textbox(label='📝 Text', lines=6, elem_classes='shiv-tb', placeholder='यहाँ text लिखें...')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b2 = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b2.click(fn=None, inputs=[b2,vd_text], outputs=vd_text, js=INSERT_TAG_JS)
                    vd_lang   = gr.Dropdown(label='🌐 Language', choices=LANG_CHOICES, value='Auto')
                    vd_speed  = gr.Slider(label='⚡ Speed',  min=0.5, max=2.0, value=1.0, step=0.05, info='0.5=Slow | 1.0=Normal | 2.0=Fast')
                    vd_pitch  = gr.Slider(label='🎵 Pitch',  min=-12,  max=12,  value=0,   step=1,    info='-12=Deep | 0=Normal | +12=High')
                    vd_energy = gr.Slider(label='💪 Energy', min=0.3, max=2.0, value=1.0, step=0.05, info='0.3=Soft | 1.0=Normal | 2.0=Loud')
                    vd_pause  = gr.Slider(label='⏸️ Pause Gap (ms)', min=0, max=500, value=0, step=50, info='Chunks ke beech silence')
                    vd_style  = gr.Textbox(label='✍️ Style Instruction (optional)', lines=2)
                    with gr.Row():
                        pc = gr.Button('😌 Calm',        size='sm')
                        pe = gr.Button('🔥 Excited',     size='sm')
                        pn = gr.Button('📺 News Anchor', size='sm')
                        ps = gr.Button('📖 Story',       size='sm')
                    vd_btn = gr.Button('🎛️ Design & Generate', variant='primary', size='lg')
                with gr.Column():
                    vd_out    = gr.Audio(label='🔊 Voice Design Output', type='numpy')
                    vd_status = gr.Textbox(label='Status', interactive=False)

            pc.click(fn=lambda:(0.8,-2,0.7,150,'speak calmly and peacefully'),                          outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            pe.click(fn=lambda:(1.3, 3,1.5,  0,'speak with excitement and high energy'),               outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            pn.click(fn=lambda:(1.0, 0,1.1,200,'speak like a professional news anchor, formal clear'), outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            ps.click(fn=lambda:(0.85,-1,0.8,250,'speak like a storyteller, warm and engaging'),        outputs=[vd_speed,vd_pitch,vd_energy,vd_pause,vd_style])
            vd_btn.click(fn_design, [vd_text,vd_lang,vd_speed,vd_pitch,vd_energy,vd_pause,vd_style], [vd_out,vd_status])

        # ── TAB 3: Simple TTS ──────────────────────────────────────────
        with gr.TabItem('🔤 Simple TTS'):
            with gr.Row():
                with gr.Column():
                    tts_text = gr.Textbox(label='📝 Text', lines=8, elem_classes='shiv-tb', placeholder='यहाँ text लिखें...')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b3 = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b3.click(fn=None, inputs=[b3,tts_text], outputs=tts_text, js=INSERT_TAG_JS)
                    tts_lang = gr.Dropdown(label='🌐 Language', choices=LANG_CHOICES, value='Auto')
                    tts_steps    = gr.Slider(label='🔢 Steps (Quality)', minimum=10, maximum=64, value=32, step=2)
                    tts_guidance = gr.Slider(label='🎯 Guidance Scale',  minimum=1.0, maximum=5.0, value=2.0, step=0.5)
                    tts_btn  = gr.Button('🔤 Generate TTS', variant='primary', size='lg')
                with gr.Column():
                    tts_out    = gr.Audio(label='🔊 Output', type='numpy')
                    tts_status = gr.Textbox(label='Status', interactive=False)
            tts_btn.click(fn_tts, [tts_text,tts_lang,tts_steps,tts_guidance], [tts_out,tts_status])

        # ── TAB 4: Instruct Mode ───────────────────────────────────────
        with gr.TabItem('📋 Instruct Mode'):
            with gr.Row():
                with gr.Column():
                    inst_text   = gr.Textbox(label='📝 Text', lines=6, elem_classes='shiv-tb', placeholder='यहाँ text लिखें...')
                    with gr.Row():
                        for tag in EVENT_TAGS:
                            b4 = gr.Button(tag, elem_classes='tag-btn', size='sm')
                            b4.click(fn=None, inputs=[b4,inst_text], outputs=inst_text, js=INSERT_TAG_JS)
                    inst_lang   = gr.Dropdown(label='🌐 Language', choices=LANG_CHOICES, value='Auto')
                    inst_prompt = gr.Textbox(label='📋 Style Instruction', lines=3, placeholder='Speak slowly and clearly...')
                    gr.Markdown('**💡 Examples:**')
                    for ex in INSTRUCT_EX:
                        eb = gr.Button(ex, size='sm')
                        eb.click(fn=lambda x=ex: x, outputs=inst_prompt)
                    inst_btn = gr.Button('📋 Generate with Instruct', variant='primary', size='lg')
                with gr.Column():
                    inst_out    = gr.Audio(label='🔊 Output', type='numpy')
                    inst_status = gr.Textbox(label='Status', interactive=False)
            inst_btn.click(fn_instruct, [inst_text,inst_lang,inst_prompt], [inst_out,inst_status])

    gr.HTML("<div style='text-align:center;padding:15px;color:#888;'>© 2026 🔱 Shiv AI Voice Cloning v3 | Shri Ram Nag | PAISAWALA</div>")

demo.launch(share=True, debug=False)
print('🔱 Shiv AI v3 launched!')